# Train a Jeff Tetris adapter on Colab (experimental)

**Not yet validated on a Colab GPU.** The draft author reported a tiny CPU dry run; repository checks exercise syntax and recovery logic with small CPU models. These checks establish neither GPU fit, speed nor gameplay quality.

For the model already published on Hugging Face, read [How the shipped Tetris adapter was trained](https://github.com/Anemll/system1-ane/blob/main/docs/TETRIS_TRAINING.md). This notebook trains a new, smaller experiment using the same dataset builder, prompt, LoRA implementation and 255-way readout.

1. Pin the trainer source and Jeff v1.3 checkpoint; verify the base weight SHA-256. Pin Transformers and FLA versions. Colab's PyTorch and other dependencies are not fully pinned; their versions are recorded.
2. Generate teacher-labeled data: 50% teacher games, 25% games with random actions, 25% garbage recovery. Reserve 256 validation and 256 test rows; audit split isolation.
3. Train Phase A on 1,024 anchors: r16, alpha 32, LoRA LR `2e-4`, readout LR `5e-6`, batch 4.
4. Continue Phase B at LoRA LR `1e-4`: 512 fresh rows, 256 anchors and up to 256 teacher-labeled states from Phase A's own games. Fresh rows fill any correction shortfall. This is supervised imitation with teacher corrections; improvement is not guaranteed.
5. Save adapter recovery archives before the separate test evaluation, then a complete archive with data, reports, helpers, checksums and notices. Optional CUDA development gameplay measures survival; ANE parity and latency require the Mac steps below.

**Runtime and memory are unmeasured.** The estimate uses an assumed `0.34` seconds per row-epoch and prints about 26 minutes. It is not a measured runtime or CU promise. The default planning target is 28 minutes, with a 4-minute allowance for final work: the trainer stops starting optimizer batches at minute 24. Setup, model loading, evaluation, saving and optional gameplay may exceed the target. This is not a hard wall-clock or billing cap. Batch 4 is a starting point, not a GPU fit guarantee.

**Differences from shipped round 02:** two 1,024-row phases starting from Jeff base, instead of continuing the pilot/best adapter through 2,048-row rounds; PyTorch correction games capped at 60 pieces; no ANE development selection. The last saved phase is retained, without claiming it is the best policy.

Reference only: the shipped Mac/MPS round-02 model reached the 150-piece cap in 30/32 ANE benchmark games, averaging 149.75 pieces and 53.75 lines. Its warm server mean was 288.20 ms/move. Those are historical measurements, not results of this notebook.

## Credits and model lineage

The shipped Tetris and Snake adapters were trained locally on an Apple M5 Max with PyTorch MPS using this repository's trainer. The head training workflow was adopted from Unsloth's notebooks; thanks to the [Unsloth team](https://github.com/unslothai/unsloth). The main path here uses ANEMLL's PyTorch LoRA trainer, without the Unsloth runtime. LoRA is a standard technique.

- **Jeff v1.3:** [Mathias Strasser / mstrasser](https://huggingface.co/mstrasser/jeff-base), based on Qwen3.5-0.8B, Apache-2.0.
- **Qwen3.5-0.8B:** [Qwen Team, Alibaba Cloud](https://huggingface.co/Qwen/Qwen3.5-0.8B), Apache-2.0.
- **Teacher labels:** [El-Tetris by Islam El-Ashi](https://imake.ninja/el-tetris-an-improvement-on-pierre-dellacheries-algorithm/), building on Pierre Dellacherie's algorithm.
- **Training, data and ANE integration:** [ANEMLL / system1-ane](https://github.com/Anemll/system1-ane).

The default-OFF Unsloth appendix is a separate, untested Tetris experiment. Its Clef joint-schema prompt/head differs from this notebook's 255-way readout and is not a drop-in replacement. Model bundles carry Apache-2.0, available upstream notices, source-license text and a notice describing this experiment.

## 0. Settings, optimization cutoff and planning estimate

Run cells in order. For a short GPU trial, use a new `RUN_ID`, `N_ANCHOR=64`, `N_FRESH=32`, `N_ANCHOR_REPLAY=16`, `N_CORR=16`, and `CORR_GAMES=2`; keep validation/test at 256. Inspect actual step times, GPU memory, correction collection and reload before scaling up. A small trial does not establish full-run fit.

Rerunning Settings preserves the original cutoff. For an intentional new experiment, restart the runtime and select a new `RUN_ID`. Existing recipe settings are checked to prevent silently reusing stale data. Batch, accumulation and evaluation batch can change for OOM recovery. A recovery from periodic weights starts a fresh optimizer and replays the dataset; it is not exact step-level resume.

`SAVE_TO_DRIVE` is optional. Automatic disconnection requires a checksum-verified Drive copy; a browser download request alone is not proof of persistence. Disconnect manually after saving if automatic disconnection is off.

In [ ]:
#@title Settings (everything below reads these)
import datetime, hashlib, json, math, os, re, shutil, signal, subprocess, sys, tarfile, textwrap, threading, time
from pathlib import Path

# Rerunning this cell keeps the original optimization deadline. Restart the runtime
# and choose a new RUN_ID to deliberately start a new experiment.
NB_START = globals().get("NB_START", time.time())

# ---- pinned sources (no secrets needed: repo and base model are public) -------------------------------
REPO_URL    = "https://github.com/Anemll/system1-ane.git"
REPO_SHA    = "acc4c69200a5d9821b92fde666f52e7247a137b1"   # trainer/source snapshot reviewed for this tutorial
BASE_REPO   = "mstrasser/jeff-base"
BASE_REV    = "3cf045d8a52dc3ac2c6d0c842071235984d5fdf5"   # tag v1.3
BASE_SHA256 = "d324dd6c9bb61b30af65564135b33f6892c30a9b2bd22667b2e09b9c8118cf77"  # model.safetensors
PIP_PACKAGES = ["transformers==5.17.0", "safetensors", "huggingface_hub", "flash-linear-attention==0.5.2"]
# torch comes from the Colab image (not reinstalled). causal-conv1d is skipped on purpose (slow source build).

# ---- experimental two-phase recipe; see docs/TETRIS_TRAINING.md for the shipped run ----
SEED       = 0       # training seed (torch + eval-sample); dataset seeds are separate, below
RANK, ALPHA = 16, 32
LORA_LR, READOUT_LR = 2e-4, 5e-6
PHASE_B_LORA_LR = 1e-4      # continuation LR used by shipped round 02
BATCH      = 4       # conservative starting micro-batch; fit/speed still need a GPU trial
GRAD_ACCUM = 1       # 1 = repo trainer unchanged. >1 = same loop with accumulation (effective batch = BATCH x GRAD_ACCUM)
ALLOW_BATCH8 = False
EPOCHS_A, EPOCHS_B = 1, 1
EVAL_BATCH, EVAL_TRAIN_ROWS = 4, 64
DEVICE = "cuda"

# ---- data (all built by the repo's projects.tetris_training.dataset builder) ----------------------------
DATA_SEED  = 42      # 42 + counts 1024/256/256 is the repo's default "pilot" configuration
N_ANCHOR   = 1024    # Phase A training rows (= the anchors that are replayed in Phase B)
N_VAL, N_TEST = 256, 256    # held-out validation (tuning/reporting) and a separate test split; both >= 256
POOL_SEED, POOL_MAX_PIECES = 20261009, 300   # fresh teacher pool for Phase B
N_FRESH, N_ANCHOR_REPLAY = 512, 256          # Phase B = 512 fresh + 256 anchors + up to 256 corrections
COLLECT_CORRECTIONS = True
N_CORR, CORR_GAMES, CORR_MAX_PIECES, CORR_SECONDS = 256, 12, 60, 180

# ---- guards ----------------------------------------------------------------------------------------------
TIME_BUDGET_MIN = 28          # planning target, NOT a hard runtime or billing limit
TAIL_SECONDS    = 240         # subtract from target: stop starting optimizer batches at minute 24
SEC_PER_ROW_EPOCH = 0.34      # planning assumption; NOT measured for this notebook
CU_PER_HOUR = None            # optionally enter the rate displayed by your Colab session
OVERRIDE_BUDGET = False       # True = allow a plan whose estimate exceeds TIME_BUDGET_MIN

# ---- outputs ---------------------------------------------------------------------------------------------
ROOT_WORK = Path("/content/tetris_colab")
RUN_ID = "trial-001"          # use a new ID when changing data or training recipe
WORK = ROOT_WORK / RUN_ID
SAVE_TO_DRIVE = False         # True = mount Drive (asks you to authorize) and copy the .tgz + sha256
DRIVE_DIR = "/content/drive/MyDrive/anemll-forge/jeff_tetris_colab"
DOWNLOAD_TO_BROWSER = False   # True = files.download() the .tgz
DISCONNECT_WHEN_DONE = False  # requires SAVE_TO_DRIVE and a verified archive copy
RUN_CUDA_GAMEPLAY = False     # optional dev games; additional time outside the planning estimate
GAMEPLAY_GAMES, GAMEPLAY_MAX_PIECES, GAMEPLAY_SECONDS = 2, 150, 600

# ---- checks ---------------------------------------------------------------------------------------------
if BATCH > 4 and not ALLOW_BATCH8:
    raise SystemExit("Start with BATCH=4 (or 2) and measure memory. Set ALLOW_BATCH8=True to try a larger batch.")
assert BATCH >= 1 and EVAL_BATCH >= 1 and GRAD_ACCUM >= 1
assert not DISCONNECT_WHEN_DONE or SAVE_TO_DRIVE, "Auto-disconnect requires a verified Drive copy"
assert N_VAL >= 256 and N_TEST >= 256, "need a held-out validation of >= 256 rows and a separate test split"
assert N_FRESH + N_ANCHOR_REPLAY + (N_CORR if COLLECT_CORRECTIONS else 0) > 0

# ---- runtime estimate (minutes) -------------------------------------------------------------------------
rows_A = N_ANCHOR * EPOCHS_A
n_corr_planned = N_CORR if COLLECT_CORRECTIONS else 0
rows_B = (N_FRESH + N_ANCHOR_REPLAY + n_corr_planned) * EPOCHS_B
EST = {
    "setup (pip, clone, 1.7 GB base download, dataset builds)": 3.0,
    "phase A: load + base/after eval (2 x (64 + 256) rows)": 2.5,
    f"phase A: train {rows_A} row-epochs": rows_A * SEC_PER_ROW_EPOCH / 60,
    "teacher-correction rollouts (capped)": (CORR_SECONDS / 60 + 1.0) if COLLECT_CORRECTIONS else 0.0,
    "phase B: load + base/after eval": 2.5,
    f"phase B: train {rows_B} row-epochs": rows_B * SEC_PER_ROW_EPOCH / 60,
    "test split eval": 1.5,
    "pack (+ optional Drive copy)": 1.0,
}
EST_TOTAL = sum(EST.values())
for k, v in EST.items():
    print(f"  {v:5.1f} min  {k}")
print(f"  {EST_TOTAL:5.1f} min  ASSUMED total; unmeasured on Colab. Planning target: {TIME_BUDGET_MIN} min.")
if CU_PER_HOUR is not None:
    print(f"  {EST_TOTAL / 60 * CU_PER_HOUR:.1f} CU estimate at the user-entered {CU_PER_HOUR} CU/h; not a cap.")
if EST_TOTAL > TIME_BUDGET_MIN and not OVERRIDE_BUDGET:
    raise SystemExit(f"Estimated {EST_TOTAL:.0f} min > TIME_BUDGET_MIN={TIME_BUDGET_MIN}. Reduce N_ANCHOR/N_FRESH/epochs "
                     "or set OVERRIDE_BUDGET=True.")

OPTIMIZATION_CUTOFF = NB_START + TIME_BUDGET_MIN * 60 - TAIL_SECONDS
def time_left_for_training():
    return OPTIMIZATION_CUTOFF - time.time()
def train_until_iso():
    return datetime.datetime.fromtimestamp(OPTIMIZATION_CUTOFF, datetime.timezone.utc).isoformat()

WORK.mkdir(parents=True, exist_ok=True)
REPO, BASE, DATA, RUN, HELP = ROOT_WORK / "system1-ane", ROOT_WORK / "jeff-base-v1.3", WORK / "data", WORK / "runs", WORK / "helpers"
for d in (DATA, RUN, HELP):
    d.mkdir(parents=True, exist_ok=True)
PY = sys.executable

def sh(cmd, log=None, cwd=None, env=None, check=True, timeout=None):
    """Stream output and preserve logs, including on failure. Optional subprocess timeout."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd[:14]) + (" ..." if len(cmd) > 14 else ""), flush=True)
    p = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, start_new_session=True)
    out = []
    def kill_group():
        try:
            os.killpg(p.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
    timer = threading.Timer(timeout, kill_group) if timeout is not None else None
    if timer:
        timer.daemon = True
        timer.start()
    try:
        for line in p.stdout:
            sys.stdout.write(line)
            out.append(line)
        rc = p.wait()
    except BaseException:
        kill_group()
        p.wait()
        raise
    finally:
        if timer:
            timer.cancel()
        p.stdout.close()
        if log:
            Path(log).write_text("".join(out))
    output = "".join(out)
    if check and rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}; see {log}")
    return rc, output


def wilson(k, n, z=1.96):
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

def fmt_acc(block):
    if not block:
        return "unavailable (evaluation incomplete)"
    n = int(block["n"]); k = round(block["accuracy"] * n); lo, hi = wilson(k, n)
    return f"{block['accuracy']:.3f}  (95% CI {lo:.3f}-{hi:.3f}, {k}/{n}, loss {block['loss']:.3f})"


recipe = {k: globals()[k] for k in (
    "REPO_SHA", "BASE_REV", "DATA_SEED", "N_ANCHOR", "N_VAL", "N_TEST", "POOL_SEED", "POOL_MAX_PIECES",
    "N_FRESH", "N_ANCHOR_REPLAY", "N_CORR", "COLLECT_CORRECTIONS", "RANK", "ALPHA", "LORA_LR",
    "PHASE_B_LORA_LR", "READOUT_LR", "EPOCHS_A", "EPOCHS_B", "SEED", "CORR_GAMES", "CORR_MAX_PIECES")}
recipe_file = WORK / "recipe.json"
if recipe_file.exists():
    assert json.loads(recipe_file.read_text()) == recipe, "Recipe changed: use a new RUN_ID; existing results are preserved"
else:
    recipe_file.write_text(json.dumps(recipe, indent=2) + "\n")

## 1. GPU check, selected dependency pins, source and base model

In [ ]:
# GPU check (subprocess: this notebook process never imports torch, so no runtime restart is needed)
rc, smi = sh(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=False)
if rc != 0:
    raise SystemExit("No GPU. Runtime > Change runtime type > A100 GPU.")
gpu_name, gpu_mem = [x.strip() for x in smi.strip().splitlines()[0].split(",")]
mem_gb = float(gpu_mem.split()[0]) / 1024
print(f"GPU: {gpu_name}, {mem_gb:.0f} GB")
if mem_gb < 38:
    print("WARNING: <38 GB. Batch 4 with ~1,500-token prompts will likely OOM here; use BATCH=2 and GRAD_ACCUM=2.")

# pinned installs (torch stays as shipped by Colab)
sh([PY, "-m", "pip", "install", "-q", *PIP_PACKAGES], log=WORK / "pip.log")
sh([PY, "-c", "import torch, transformers, fla; print('torch', torch.__version__, '| transformers', transformers.__version__, "
              "'| fla', getattr(fla, '__version__', '?'), '| cuda', torch.cuda.is_available())"])

# pinned repo source at one commit
if not REPO.exists():
    sh(["git", "clone", "-q", REPO_URL, REPO])
sh(["git", "-C", REPO, "checkout", "-q", REPO_SHA])
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == REPO_SHA, head
print("repo at", head)

# pinned base model, sha256 verified (public repo: no HF token needed)
from huggingface_hub import snapshot_download
snapshot_download(BASE_REPO, revision=BASE_REV, local_dir=str(BASE))
h = hashlib.sha256()
with open(BASE / "model.safetensors", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 24), b""):
        h.update(chunk)
assert h.hexdigest() == BASE_SHA256, f"base checkpoint sha256 mismatch: {h.hexdigest()}"
cfg = json.loads((BASE / "config.json").read_text())
types = cfg["text_config"]["layer_types"]
print("base verified:", BASE_REV[:8], "| arch", cfg["architectures"], "| layers", len(types),
      f"({types.count('linear_attention')} Gated DeltaNet + {types.count('full_attention')} full attention)")


## 2. Note on re-keying the base

**About "re-keying" the base.** The repo's trainer (`scripts/jeff_lora_train.py`) loads the published Jeff checkpoint **as is** (`AutoModel` -> `Qwen3_5Model`, keys `language_model.*`), and its merge step looks up exactly those key names; this is how the Oct 7 Colab continuation and the shipped Mac runs loaded it. The re-key to the `Qwen3_5ForConditionalGeneration` layout (`model.language_model.*`) exists only in the *Unsloth* Snake recipe (`run_decision_a100.py: jeff_folder()`), because Unsloth's decision head reads the backbone's output embeddings. So the main path below does **not** re-key (re-keying would break the repo's merge). The Unsloth appendix reuses that re-key function.

## 3. Dataset: the repo's builder (CPU, stdlib only)

- **main** build, `--seed 42 --train 1024 --validation 256 --test 256`: this is the repo's default pilot configuration. Its `train` split is the **anchor** set (Phase A rows and Phase B replay); `validation` is the held-out set used for before/after; `test` is touched once at the end. The builder also reserves 8 dev seeds, 32 test seeds and 32 correction-collection seeds (disjoint from every game that produced rows).
- **pool** build (different seed, games up to 300 pieces): fresh teacher rows for Phase B, built with `--exclude-dataset` for every main row, so nothing overlaps validation, test or the anchors.
- Both builds are audited by the repo's `audit` command (recomputes labels, legal placements, prompt text, option order, split isolation, checksums).
- Row mixture per split: 50% teacher games, 25% games with 10% uniformly random actions (label is still the teacher's best placement), 25% teacher recovery from 4-10 rows of synthetic garbage. Singleton-option decisions are excluded.

In [ ]:
def dataset_cmd(*args):
    return [PY, "-m", "projects.tetris_training.dataset", *args]

MAIN, POOL = DATA / "main", DATA / "pool"
if not MAIN.exists():
    sh(dataset_cmd("build", "--output", MAIN, "--train", N_ANCHOR, "--validation", N_VAL, "--test", N_TEST,
                   "--seed", DATA_SEED), cwd=REPO, log=WORK / "build_main.log")
sh(dataset_cmd("audit", "--dataset", MAIN), cwd=REPO, log=WORK / "audit_main.log")
if not POOL.exists():
    sh(dataset_cmd("build", "--output", POOL, "--train", N_FRESH + N_CORR + N_ANCHOR_REPLAY, "--validation", 4, "--test", 4,
                   "--seed", POOL_SEED, "--max-pieces", POOL_MAX_PIECES,
                   "--exclude-dataset", MAIN / "train.jsonl", "--exclude-dataset", MAIN / "validation.jsonl",
                   "--exclude-dataset", MAIN / "test.jsonl"), cwd=REPO, log=WORK / "build_pool.log")
sh(dataset_cmd("audit", "--dataset", POOL), cwd=REPO, log=WORK / "audit_pool.log")

sys.path.insert(0, str(REPO))           # repo helpers (pure Python, no torch)
from projects.tetris_training.dataset import balanced_sample, read_jsonl, sha256 as file_sha256, state_key, write_jsonl
from collections import Counter
anchors, val_rows, test_rows = (read_jsonl(MAIN / f"{s}.jsonl") for s in ("train", "validation", "test"))
pool = read_jsonl(POOL / "train.jsonl")
held_keys = {state_key(r) for r in val_rows + test_rows}
assert not held_keys & {state_key(r) for r in anchors}, "anchors overlap held-out"
assert not held_keys & {state_key(r) for r in pool}, "pool overlaps held-out"
assert not {state_key(r) for r in anchors} & {state_key(r) for r in pool}, "pool overlaps anchors"
mean_opts = sum(len(r["options"]) for r in val_rows) / len(val_rows)
print(f"validation n={len(val_rows)}  test n={len(test_rows)}  anchors n={len(anchors)}  fresh pool n={len(pool)}")
print("anchor modes:", dict(Counter(r["source"]["mode"] for r in anchors)))
random_agreement = sum(1 / len(r["options"]) for r in val_rows) / len(val_rows)
print(f"mean legal options {mean_opts:.1f}; random expected agreement {random_agreement:.1%}")
ex = val_rows[0]
print("\nprompt option format (identical to serving):\n  ", list(ex["options"].items())[0])
print("  board rows:", len(ex["state"]["latest"]["board"].splitlines()), "x", len(ex["state"]["latest"]["board"].splitlines()[0]),
      "| piece", ex["state"]["latest"]["piece"], "| options", len(ex["options"]), "(<= 255 readout codes)")

## 4. Phase A: anchor training and recoverable attempts

This cell writes two helpers: `run_train.py` adds optional accumulation to the repo's training loop; `collect_corrections.py` runs the same Tetris game and prompt directly through PyTorch, for collection or evaluation.

Phase A uses rank 16, alpha 32, no dropout, LoRA LR `2e-4`, readout LR `5e-6`, AdamW without weight decay, gradient clipping at 1.0 and bf16. `--heldout-dataset` keeps all training rows available for training. `--skip-merge` saves adapter weights for later Mac export. The optimization cutoff is checked between batches; it does not bound evaluation or total runtime.

Each attempt gets a fresh output directory. Rerunning reuses a final saved adapter or continues from the latest recovery weights in a new attempt. Periodic recovery occurs every 64 optimizer steps. If OOM happens before the first save, retry starts from the phase's original initialization. Reduce **both** training and evaluation batches if needed; accumulation affects training only. Each usable result gets an immediate recovery archive, including after post-training evaluation fails.

In [ ]:
(HELP / "run_train.py").write_text(r'''"""Run the repo trainer (scripts/jeff_lora_train.py) with optional gradient accumulation.

usage: python run_train.py REPO_DIR GRAD_ACCUM [trainer args...]

GRAD_ACCUM == 1 runs the repo trainer unchanged. GRAD_ACCUM > 1 swaps ONLY the training loop of
jeff_lora_train.run_epoch for an accumulating one (same loss, same AdamW groups, same grad-clip 1.0,
one optimizer step per GRAD_ACCUM micro-batches of --batch rows). Evaluation is untouched.
"""
import math
import sys
import time
from pathlib import Path

repo, accum, argv = Path(sys.argv[1]), int(sys.argv[2]), sys.argv[3:]
sys.path.insert(0, str(repo / "scripts"))
sys.path.insert(0, str(repo / "coreai"))
import torch  # noqa: E402
import jeff_lora_train as T  # noqa: E402

if accum > 1:
    _orig = T.run_epoch

    def run_epoch_accum(model, readout, rows, batch_size, pad_id, device, optimizer=None,
                        *, stop_at=None, step_callback=None):
        if optimizer is None:
            return _orig(model, readout, rows, batch_size, pad_id, device, None,
                         stop_at=stop_at, step_callback=step_callback)
        model.train(False)
        readout.train(True)
        params = [p for g in optimizer.param_groups for p in g["params"]]
        order = torch.randperm(len(rows)).tolist()
        n_micro = math.ceil(len(order) / batch_size)
        total_loss, correct, seen, steps, micro, secs, stopped = 0.0, 0, 0, 0, 0, [], False
        optimizer.zero_grad(set_to_none=True)
        began = time.perf_counter()
        for start in range(0, len(order), batch_size):
            if micro % accum == 0:  # only stop on an optimizer-step boundary
                if stop_at is not None and time.time() >= stop_at:
                    stopped = True
                    print("training deadline reached; saving completed work", flush=True)
                    break
                began = time.perf_counter()
            group = min(accum, n_micro - (micro // accum) * accum)
            group_start = (micro // accum) * accum * batch_size
            group_rows = min(accum * batch_size, len(order) - group_start)
            batch = [rows[i] for i in order[start:start + batch_size]]
            ids, mask, index, labels = T.collate(batch, pad_id, device)
            with torch.enable_grad():
                logits = T.masked_logits(T.forward_logits(model, readout, ids, mask, index, device), batch)
                loss = torch.nn.functional.cross_entropy(logits, labels)
                (loss * (len(batch) / group_rows)).backward()
            total_loss += float(loss.detach()) * len(batch)
            correct += int((logits.detach().argmax(-1) == labels).sum())
            seen += len(batch)
            micro += 1
            if micro % accum == 0 or micro == n_micro:
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                T.synchronize(device)
                steps += 1
                secs.append(time.perf_counter() - began)
                if steps <= 2 or steps % 8 == 0:
                    print(f"    train step {steps} (x{group} micro-batches of {batch_size})  {secs[-1]:.2f}s  "
                          f"loss {total_loss / seen:.3f}  acc {correct / seen:.3f}", flush=True)
                if step_callback is not None:
                    step_callback(steps, seen)
        optimizer.zero_grad(set_to_none=True)
        steady = secs[1:] or secs
        return {"loss": total_loss / max(1, seen), "accuracy": correct / max(1, seen), "n": seen, "steps": steps,
                "seconds_per_step": round(sum(steady) / max(1, len(steady)), 3), "stopped_early": stopped,
                "grad_accum": accum}

    T.run_epoch = run_epoch_accum
    print(f"run_train: gradient accumulation x{accum} (effective batch = --batch x {accum})", flush=True)
raise SystemExit(T.main(argv))
''')
(HELP / "collect_corrections.py").write_text(r'''"""Teacher corrections from the trained model's OWN boards (PyTorch, no ANE server).

Same recipe as projects/tetris_training/play.py --collect: the current adapter plays greedy games on the dataset's
reserved collection seeds (losing games included); every visited board is labeled by El-Tetris, not by the model.

usage: python collect_corrections.py REPO MODEL ADAPTER DATASET OUT_DIR --exclude a.jsonl [--exclude b.jsonl ...]
"""
import argparse
import json
import statistics
import sys
import time
from pathlib import Path

p = argparse.ArgumentParser()
p.add_argument("repo", type=Path)
p.add_argument("model", type=Path)
p.add_argument("adapter", type=Path)
p.add_argument("dataset", type=Path, help="audited main dataset dir (provides reserved collection seeds)")
p.add_argument("out", type=Path)
p.add_argument("--exclude", type=Path, action="append", default=[])
p.add_argument("--games", type=int, default=12)
p.add_argument("--max-pieces", type=int, default=60)
p.add_argument("--max-seconds", type=float, default=240.0, help="stop starting new games after this wall time")
p.add_argument("--device", default="cuda")
p.add_argument("--split", choices=("collect", "dev", "test"), default="collect")
a = p.parse_args()
started = time.time()
sys.path.insert(0, str(a.repo))
sys.path.insert(0, str(a.repo / "scripts"))
sys.path.insert(0, str(a.repo / "coreai"))
import torch  # noqa: E402
from safetensors.torch import load_file  # noqa: E402
from torch import nn  # noqa: E402
from transformers import AutoModel, AutoTokenizer  # noqa: E402
import jeff_lora_train as T  # noqa: E402
from jeff_coreai import load_decision_config  # noqa: E402
from jeff_lora import attach_lora  # noqa: E402
from jeff_tetris import tetris_row  # noqa: E402
from projects.tetris_training.dataset import read_jsonl, rollout, teacher_pick, write_jsonl  # noqa: E402
from projects.tetris_training.play import deduplicate_corrections  # noqa: E402

model, adapter, device = a.model.resolve(), a.adapter.resolve(), a.device
cfg = json.loads((adapter / "adapter_config.json").read_text())
decision = load_decision_config(model)
tokenizer = AutoTokenizer.from_pretrained(str(model))
pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else (tokenizer.eos_token_id or 0)
dtype = torch.bfloat16 if device in ("cuda", "mps") else torch.float32
backbone = AutoModel.from_pretrained(str(model), dtype=dtype, attn_implementation="sdpa").to(device)
backbone.requires_grad_(False)
backbone.eval()
layers = attach_lora(backbone, int(cfg["rank"]), float(cfg["scale"]) * int(cfg["rank"]))
w = load_file(str(model / "readout.safetensors"))["weight"]
readout = nn.Linear(int(w.shape[1]), int(w.shape[0]), bias=False)
T.load_init_adapter(adapter, layers, readout)
readout.to(device)


def choose(board, piece, options):
    if time.time() - started >= a.max_seconds:
        raise TimeoutError("gameplay deadline reached")
    row = tetris_row(board, piece, teacher_pick(options)["key"], options)  # label unused; shares the serving prompt
    enc = T.encode(model, [row], decision, tokenizer)
    ids, mask, index, _ = T.collate(enc, pad_id, device)
    with torch.no_grad():
        logits = T.masked_logits(T.forward_logits(backbone, readout, ids, mask, index, device), enc)
    return list(row["options"])[int(logits.argmax(-1)[0])]


manifest = json.loads((a.dataset / "manifest.json").read_text())
seeds = (manifest["collection_seeds"] if a.split == "collect" else manifest["evaluation_seeds"][a.split])[:a.games]
games, collected = [], []
existing = [r for path in a.exclude for r in read_jsonl(path)]
a.out.mkdir(parents=True, exist_ok=True)

def save_progress():
    corrections = deduplicate_corrections(collected, existing) if a.split == "collect" else []
    if a.split == "collect":
        temporary_rows = a.out / "corrections.jsonl.tmp"
        write_jsonl(temporary_rows, corrections)
        temporary_rows.replace(a.out / "corrections.jsonl")
    report = {"games": games, "requested_games": len(seeds), "completed_games": len(games),
              "visited_states": len(collected), "unique_new_corrections": len(corrections),
              "max_pieces": a.max_pieces, "seconds": round(time.time() - started, 1), "device": device,
              "split": a.split, "backend": "PyTorch (not ANE)",
              "summary": {"games": len(games), "pieces_mean": statistics.mean(g["pieces"] for g in games),
                          "lines_mean": statistics.mean(g["lines"] for g in games),
                          "capped": sum(g["capped"] for g in games)} if games else None,
              "note": "TRAINING collection seeds; not an evaluation" if a.split == "collect" else
                      "Fixed-seed PyTorch gameplay; incomplete if completed_games < requested_games. No ANE latency claim."}
    temporary = a.out / "report.json.tmp"
    temporary.write_text(json.dumps(report, indent=2) + "\n")
    temporary.replace(a.out / "report.json")
    return report

save_progress()
for seed in seeds:
    try:
        rows, game = rollout(seed, "jeff:colab", a.max_pieces, 0, f"{a.split}:play:{seed}", choose=choose)
    except TimeoutError:
        print("Time cap reached; retaining completed games only", flush=True)
        break
    games.append(game)
    if a.split == "collect":
        collected.extend(rows)
    save_progress()
    print(f"game {len(games)}/{len(seeds)}: {game['pieces']} pieces, {game['lines']} lines, {game['terminal']}", flush=True)
print(json.dumps(save_progress()["summary"]), flush=True)

''')

def trainer_args(output, train_jsonl, epochs, lora_lr, init_adapter=None):
    args = ["--model", BASE, "--output", output, "--task", "tetris",
            "--dataset", train_jsonl, "--heldout-dataset", MAIN / "validation.jsonl",
            "--device", DEVICE, "--rank", RANK, "--alpha", ALPHA, "--epochs", epochs, "--batch", BATCH,
            "--lr", lora_lr, "--readout-lr", READOUT_LR, "--seed", SEED,
            "--eval-batch", EVAL_BATCH, "--eval-train-rows", EVAL_TRAIN_ROWS,
            "--skip-merge", "--checkpoint-every", 64, "--train-until", train_until_iso()]
    if init_adapter:
        args += ["--init-adapter", init_adapter]
    return args

def trainer_cmd(args):
    if GRAD_ACCUM == 1:
        return [PY, REPO / "forge.py", "jeff-train-lora", *args]
    return [PY, HELP / "run_train.py", REPO, GRAD_ACCUM, *args]

def adapter_ready(path):
    return all((path / f).is_file() and (path / f).stat().st_size > 0 for f in
               ("adapter_model.safetensors", "readout.safetensors", "adapter_config.json"))

def saved_candidate(output):
    report_file = output / "report.json"
    report = json.loads(report_file.read_text()) if report_file.exists() else {}
    adapter = output / "adapter"
    if not adapter_ready(adapter):
        pointer = output / "latest-recovery.json"
        if not pointer.exists():
            return None
        adapter = Path(json.loads(pointer.read_text())["adapter"])
        if not adapter_ready(adapter):
            return None
    return {**report, "_output": str(output), "_adapter": str(adapter),
            "_partial": report.get("save_state") != "complete" or any(h.get("stopped_early") for h in report.get("history", []))}

def write_archive(destination, entries):
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    with tarfile.open(temporary, "w:gz") as archive:
        for source, name in entries:
            archive.add(source, arcname=name)
    temporary.replace(destination)
    digest = hashlib.sha256(destination.read_bytes()).hexdigest()
    destination.with_name(destination.name + ".sha256").write_text(f"{digest}  {destination.name}\n")
    return digest

def backup_adapter(name, candidate):
    entries = [(Path(candidate["_adapter"]), "adapter"), (WORK / "recipe.json", "recipe.json")]
    for filename in ("report.json", "latest-recovery.json"):
        source = Path(candidate["_output"]) / filename
        if source.exists():
            entries.append((source, filename))
    dest = WORK / f"{name}-recovery.tgz"
    write_archive(dest, entries)
    print("Adapter recovery archive (available even if later evaluation fails):", dest)

def run_phase(name, train_jsonl, epochs, lora_lr, init_adapter=None):
    parent = RUN / name
    parent.mkdir(exist_ok=True)
    attempts = sorted(parent.glob("attempt-*"))
    previous = next((c for out in reversed(attempts) if (c := saved_candidate(out))), None)
    # A final adapter is already published even if its subsequent evaluation failed.
    if previous and Path(previous["_adapter"]).name == "adapter":
        backup_adapter(name, previous)
        print(f"[{name}] reusing saved adapter; no repeated optimizer steps")
        return previous
    if time_left_for_training() < 120:
        print(f"[optimization cutoff] skipping {name}")
        if previous:
            backup_adapter(name, previous)
        return previous
    if previous:
        init_adapter = previous["_adapter"]
        print("Recovering LoRA/readout weights with a fresh optimizer; this attempt replays the dataset.")
    output = parent / f"attempt-{len(attempts) + 1:04d}"
    rc, log_text = sh(trainer_cmd(trainer_args(output, train_jsonl, epochs, lora_lr, init_adapter)),
                      cwd=REPO, log=WORK / f"train_{name}_{output.name}.log", check=False)
    candidate = saved_candidate(output) or previous
    if candidate:
        backup_adapter(name, candidate)
    if rc != 0:
        print(f"[{name}] training/evaluation failed (exit {rc}); saved candidate: {bool(candidate)}")
        if "out of memory" in log_text.lower():
            print("For retry, set BATCH=2, GRAD_ACCUM=2 and EVAL_BATCH=2. Rerun Settings then this phase. "
                  "A checkpoint-only retry uses a fresh output and optimizer; final saved adapters are reused.")
    if candidate:
        history = candidate.get("history", [])
        done = sum(h["n"] for h in history)
        seconds = sum(h.get("seconds", 0) for h in history)
        print(f"[{name}] report records {done} row-epochs in {seconds / 60:.1f} min; partial={candidate['_partial']}")
        print("held-out before:", fmt_acc(candidate.get("base", {}).get("heldout")))
        print("held-out after: ", fmt_acc(candidate.get("adapter", {}).get("heldout")))
    return candidate

write_jsonl(RUN / "phaseA_train.jsonl", anchors)
repA = run_phase("phaseA", RUN / "phaseA_train.jsonl", EPOCHS_A, LORA_LR)

## 5. Teacher corrections and Phase B data

Phase A plays greedily on reserved **training collection seeds**. Losing games are included. El-Tetris labels the visited boards; the model's actions are not used as targets. Games are capped at 60 pieces by default. The collection subprocess has a timeout including loading and retains completed games; it may return fewer or no corrections.

Phase B contains up to 1/4 corrections, 1/4 anchor replay and fresh teacher data for the rest. Rows are balanced by piece, stack height and game stage, deduplicated and checked against held-out states. Once created, the Phase B mixture is retained unchanged for retries.

In [ ]:
if (RUN / "phaseB_train.jsonl").exists():
    mixB = RUN / "phaseB_train.jsonl"
    existing_mix = read_jsonl(mixB)
    assert not {state_key(r) for r in existing_mix} & held_keys
    print("Reusing original Phase B data for retry:", mixB)
else:
    corr_rows = []
    if repA is not None and COLLECT_CORRECTIONS and time_left_for_training() > CORR_SECONDS + 400:
        CORR = RUN / "corrections"
        shutil.rmtree(CORR, ignore_errors=True)
        sh([PY, HELP / "collect_corrections.py", REPO, BASE, repA["_adapter"], MAIN, CORR,
            "--exclude", MAIN / "train.jsonl", "--exclude", MAIN / "validation.jsonl", "--exclude", MAIN / "test.jsonl",
            "--exclude", POOL / "train.jsonl", "--games", CORR_GAMES, "--max-pieces", CORR_MAX_PIECES,
            "--max-seconds", CORR_SECONDS, "--device", DEVICE], cwd=REPO, log=WORK / "collect.log", check=False, timeout=CORR_SECONDS + 5)
        corr_rows = [r for r in read_jsonl(CORR / "corrections.jsonl") if state_key(r) not in held_keys] if (CORR / "corrections.jsonl").exists() else []
        print(f"unique teacher-labeled correction states: {len(corr_rows)}")
    elif repA is not None and COLLECT_CORRECTIONS:
        print("[time guard] not enough time left for correction rollouts; Phase B will use fresh + anchor rows only")
    
    mixB = None
    if repA is not None:
        n_total = N_FRESH + N_ANCHOR_REPLAY + (N_CORR if COLLECT_CORRECTIONS else 0)
        n_corr = min(N_CORR, len(corr_rows))
        chosen = balanced_sample(corr_rows, n_corr, 11) if n_corr else []
        chosen += balanced_sample(anchors, N_ANCHOR_REPLAY, 12)
        seen = {state_key(r) for r in chosen}
        available = [r for r in pool if state_key(r) not in seen]
        chosen += balanced_sample(available, n_total - len(chosen), 13)      # fresh (also fills any correction shortfall)
        import random
        random.Random(14).shuffle(chosen)
        keys = [state_key(r) for r in chosen]
        assert len(keys) == len(set(keys)) == n_total, "duplicate states in the Phase B mixture"
        assert not set(keys) & held_keys, "Phase B mixture overlaps validation/test"
        if (RUN / "phaseB_train.jsonl").exists():
            print("Reusing the original Phase B dataset for retries.")
            chosen = read_jsonl(RUN / "phaseB_train.jsonl")
        else:
            write_jsonl(RUN / "phaseB_train.jsonl", chosen)
        mixB = RUN / "phaseB_train.jsonl"
        print(f"Phase B mixture: {len(chosen)} rows = {n_corr} corrections + {N_ANCHOR_REPLAY} anchors + "
              f"{len(chosen) - n_corr - N_ANCHOR_REPLAY} fresh | modes {dict(Counter(r['source']['mode'] for r in chosen))}")
        print("sha256", file_sha256(mixB))

## 6. Phase B: supervised continuation

Continue from Phase A's saved LoRA/readout at LoRA LR **`1e-4`**, with a fresh optimizer. Readout LR remains `5e-6`. If no Phase B weights are saved, use Phase A. Partial checkpoints are explicitly labeled. This tutorial retains the latest saved phase; it does not select the strongest checkpoint by gameplay.

In [ ]:
repB = None
if mixB is not None:
    repB = run_phase("phaseB", mixB, EPOCHS_B, PHASE_B_LORA_LR, init_adapter=repA["_adapter"])
FINAL = "phaseB" if repB is not None else ("phaseA" if repA is not None else None)
assert FINAL, "No usable weights saved. Check logs; lower train/eval batch sizes before retrying."
final_report = repB or repA
final_adapter = Path(final_report["_adapter"])
print("final adapter:", final_adapter, "| partial:", final_report["_partial"])
print("This tutorial retains the last saved phase; it has not selected a winner by gameplay.")

## 7. Held-out teacher agreement

Validation before/after uses the same 256 rows; a separate 256-row test is scored with the final adapter. Before-base accuracy is unavailable if recovery replaced the original initialization and no original report survives. Incomplete evaluations remain `null`; they do not prevent packing saved weights.

Wilson 95% intervals describe row-level binomial proportions. Rows from the same game are correlated, so these intervals are not a guarantee of independent generalization or a significance test for before/after improvement. Agreement with the teacher is different from survival and lines cleared.

In [ ]:
def eval_only(name, jsonl, adapter=None):
    args = ["--model", BASE, "--output", RUN / f"_unused_{name}", "--task", "tetris", "--dataset", MAIN / "validation.jsonl",
            "--heldout-dataset", jsonl, "--device", DEVICE, "--rank", RANK, "--alpha", ALPHA, "--eval-batch", EVAL_BATCH,
            "--eval-only", "--seed", SEED]
    if adapter:
        args += ["--init-adapter", adapter]
    rc, text = sh([PY, REPO / "forge.py", "jeff-train-lora", *args], cwd=REPO, log=WORK / f"eval_{name}.log", check=False)
    m = re.search(r"eval heldout: acc ([0-9.]+)\s+loss ([0-9.]+)\s+n (\d+)", text)
    if rc != 0 or not m:
        raise RuntimeError(f"eval {name} failed")
    return {"accuracy": float(m[1]), "loss": float(m[2]), "n": int(m[3])}

test_after, eval_error = None, None
try:
    test_after = eval_only("test_after", MAIN / "test.jsonl", final_adapter)
except Exception as exc:
    eval_error = str(exc)
    print("Test evaluation incomplete; continuing to package the saved adapter:", eval_error)

results = {
    "validation_before_base": repA.get("base", {}).get("heldout") if not repA.get("init_adapter") else None,
    "validation_after_phaseA": repA.get("adapter", {}).get("heldout"),
    "validation_after_final": final_report.get("adapter", {}).get("heldout"),
    "test_after_final": test_after,
    "final_phase": FINAL, "partial": final_report["_partial"], "test_error": eval_error,
    "final_adapter": str(final_adapter), "base_sha256": BASE_SHA256,
    "settings": {k: globals()[k] for k in ("SEED", "RANK", "ALPHA", "LORA_LR", "READOUT_LR", "PHASE_B_LORA_LR", "BATCH", "GRAD_ACCUM",
                                          "EPOCHS_A", "EPOCHS_B", "DATA_SEED", "N_ANCHOR", "N_VAL", "N_TEST", "POOL_SEED",
                                          "N_FRESH", "N_ANCHOR_REPLAY", "N_CORR", "COLLECT_CORRECTIONS", "REPO_SHA", "BASE_REV")},
    "wall_minutes_so_far": round((time.time() - NB_START) / 60, 1),
    "gpu": gpu_name,
}
(RUN / "results.json").write_text(json.dumps(results, indent=2, default=str) + "\n")

print("\n=== Held-out teacher agreement; unavailable fields mean evaluation did not complete ===")
print("before (Jeff base readout)  ", fmt_acc(results["validation_before_base"]))
print("after phase A               ", fmt_acc(results["validation_after_phaseA"]))
if repB is not None:
    print("after phase B (final)       ", fmt_acc(results["validation_after_final"]))
print("test split, final adapter   ", fmt_acc(test_after))
print("\nREFERENCE ONLY, shipped round 02 (Mac, MPS, several 2,048-row rounds, other rows): 0.684 -> 0.746 on 256 rows;")
print("ANE play: 30/32 capped, 149.75 pieces, 53.75 lines, 288.20 ms/move. Not comparable with this unmeasured Colab experiment.")
print(f"\nwall time since the clock started: {(time.time() - NB_START) / 60:.1f} min (planning target {TIME_BUDGET_MIN}; not a hard runtime cap)")

### Optional CUDA gameplay on development seeds

Set `RUN_CUDA_GAMEPLAY=True` to run 2 development games (150-piece cap). This adds up to the configured subprocess timeout after model loading is included. Completed games and an explicit requested/completed count are saved. It is a small policy check, not a 32-seed benchmark or ANE performance measurement. Use development seeds for tuning; reserve test seeds for a final comparison. The exported helper also supports `--split test --games 32 --max-pieces 150` on CUDA with a suitable timeout.


In [ ]:
if RUN_CUDA_GAMEPLAY:
    destination = RUN / "cuda-gameplay"
    destination.mkdir(exist_ok=True)
    rc, _ = sh([PY, HELP / "collect_corrections.py", REPO, BASE, final_adapter, MAIN, destination,
                "--split", "dev", "--games", GAMEPLAY_GAMES, "--max-pieces", GAMEPLAY_MAX_PIECES,
                "--max-seconds", GAMEPLAY_SECONDS, "--device", DEVICE], cwd=REPO,
               log=WORK / "cuda-gameplay.log", check=False, timeout=GAMEPLAY_SECONDS + 5)
    report_path = destination / "report.json"
    results["cuda_gameplay"] = json.loads(report_path.read_text()) if report_path.exists() else None
    results["cuda_gameplay_exit"] = rc
    (RUN / "results.json").write_text(json.dumps(results, indent=2, default=str) + "\n")
    print(json.dumps(results["cuda_gameplay"], indent=2))
else:
    print("CUDA gameplay skipped; enable RUN_CUDA_GAMEPLAY for a development policy check.")


## 8. Export a bundle and preserve it

The archive contains final adapter weights, reports (including missing-evaluation status), main dataset and its gameplay seeds, Phase B rows, corrections, helper source, recipe, versions, source/model notices, and optional CUDA gameplay results. A SHA-256 sidecar accompanies it. Smaller recovery archives from each phase already exist if this cell is never reached.

Download **both** the archive and checksum, or enable a verified Drive copy. Data under `/content` disappears when the runtime is deleted. Auto-disconnect is allowed only after Drive copy verification.

In [ ]:
B = WORK / "bundle" / "jeff_tetris_colab"
shutil.rmtree(B, ignore_errors=True)
B.mkdir(parents=True)
shutil.copytree(final_adapter, B / "adapter")
for ph, report in (("phaseA", repA), ("phaseB", repB)):
    if report:
        (B / "reports").mkdir(exist_ok=True)
        (B / "reports" / f"{ph}_report.json").write_text(json.dumps(report, indent=2) + "\n")
        parity = Path(report["_output"]) / "parity_rows.json"
        if parity.exists():
            shutil.copy(parity, B / "reports" / f"{ph}_parity_rows.json")
shutil.copytree(MAIN, B / "dataset_main")
for f in ("phaseB_train.jsonl",):
    if (RUN / f).exists():
        shutil.copy(RUN / f, B / f)
if (RUN / "corrections").exists():
    shutil.copytree(RUN / "corrections", B / "corrections")
shutil.copy(RUN / "results.json", B / "results.json")
shutil.copy(WORK / "recipe.json", B / "recipe.json")
shutil.copy(POOL / "manifest.json", B / "pool_manifest.json")
shutil.copytree(HELP, B / "helpers")
if (RUN / "cuda-gameplay").exists():
    shutil.copytree(RUN / "cuda-gameplay", B / "cuda-gameplay")
# Model weights use Apache-2.0; preserve the model's own notices where supplied.
shutil.copy(REPO / "hf" / "LICENSE", B / "LICENSE")
shutil.copy(REPO / "LICENSE", B / "SOURCE_LICENSE")
(B / "upstream_notices").mkdir()
for source in BASE.iterdir():
    if source.is_file() and (source.name.upper().startswith(("LICENSE", "NOTICE")) or source.name == "README.md"):
        shutil.copy(source, B / "upstream_notices" / source.name)
(B / "NOTICE").write_text(
    "Experimental Jeff Tetris LoRA plus readout; generated by the Colab tutorial, not the shipped round-02 model.\n"
    "Base: Jeff v1.3 by Mathias Strasser (https://huggingface.co/mstrasser/jeff-base), derived from\n"
    "Qwen3.5-0.8B by Qwen Team, Alibaba Cloud (https://huggingface.co/Qwen/Qwen3.5-0.8B). Apache-2.0.\n"
    "Modifications: LoRA and readout fine-tuning by the notebook operator using ANEMLL's PyTorch trainer.\n"
    "Source: https://github.com/Anemll/system1-ane (MIT). Labels: El-Tetris by Islam El-Ashi,\n"
    "building on Pierre Dellacherie's algorithm. Head training workflow adopted from Unsloth notebooks;\n"
    "the main training path does not use the Unsloth runtime. See recipe.json and results.json for provenance.\n")
for log in WORK.glob("*.log"):
    (B / "logs").mkdir(exist_ok=True)
    shutil.copy(log, B / "logs" / log.name)
versions = subprocess.run([PY, "-m", "pip", "list", "--format=freeze"], capture_output=True, text=True).stdout
(B / "versions.txt").write_text(f"repo {REPO_SHA}\nbase {BASE_REPO}@{BASE_REV}\ngpu {gpu_name}\n\n" + "\n".join(
    l for l in versions.splitlines() if l.split("==")[0].lower() in ("torch", "transformers", "safetensors", "flash-linear-attention", "triton", "huggingface-hub")) + "\n")
(B / "NEXT_STEPS_MAC.txt").write_text("Optional, Apple-silicon Mac only: see section 9 of 03_tetris_training_colab.ipynb "
    "(merge_saved_lora -> jeff-convert -> compile -> system1-ane-serve -> projects.tetris_training.play on dataset_main).\n")
tgz = WORK / "jeff_tetris_colab_adapter_bundle.tgz"
digest = write_archive(tgz, [(B, "jeff_tetris_colab")])
print(f"{tgz}  {tgz.stat().st_size / 1e6:.1f} MB\nsha256 {digest}")

drive_verified = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = Path(DRIVE_DIR); dest.mkdir(parents=True, exist_ok=True)
    shutil.copy(tgz, dest / tgz.name); shutil.copy(WORK / (tgz.name + ".sha256"), dest / (tgz.name + ".sha256"))
    assert hashlib.sha256((dest / tgz.name).read_bytes()).hexdigest() == digest, "Drive copy hash mismatch"
    drive_verified = True
    print("copied to Drive and verified:", dest / tgz.name)
if DOWNLOAD_TO_BROWSER:
    from google.colab import files
    files.download(str(tgz))
    files.download(str(tgz.with_name(tgz.name + ".sha256")))
print(f"\nTOTAL wall time {(time.time() - NB_START) / 60:.1f} min. Runtime > Disconnect and delete runtime when you are done (or set DISCONNECT_WHEN_DONE=True).")
if DISCONNECT_WHEN_DONE:
    assert drive_verified, "Runtime retained: no verified persistent archive copy"
    from google.colab import runtime
    runtime.unassign()

## 9. OPTIONAL, Apple-silicon Mac only: ANE playback and the 32-seed evaluation

**This cannot run on Colab** (Core AI compile/serve exist only on macOS). The optional CUDA games above measure the PyTorch policy. They do not establish ANE parity, placement or latency. Use the selected training Python for merge and evaluation, and the Core AI SDK interpreter for convert/compile/serve. Check a spare port before starting the server.

```sh
# on the Mac, inside a checkout of Anemll/system1-ane at the commit pinned above
shasum -a 256 -c jeff_tetris_colab_adapter_bundle.tgz.sha256        # in the folder holding both downloaded files
export OUT=${OUT:-$PWD/jeff-tetris-colab-out}; mkdir -p "$OUT"; tar xzf jeff_tetris_colab_adapter_bundle.tgz -C "$OUT"
export JEFF_BASE_MODEL=${JEFF_BASE_MODEL:?set to your mstrasser/jeff-base @ 3cf045d8 checkout}   # same bytes as the Colab base
python -c "import sys; sys.path.insert(0,'coreai'); from pathlib import Path; from jeff_lora import merge_saved_lora; import os; \
  o=os.environ['OUT']; merge_saved_lora(Path(os.environ['JEFF_BASE_MODEL']), Path(o)/'jeff_tetris_colab/adapter', Path(o)/'merged')"
export COREAI_PYTHON=/path/to/coreai-sdk/bin/python
python forge.py jeff-convert --model "$OUT/merged" --output "$OUT/build"      # defaults (--prefill 256 --ctx 2048) match the shipped p256_2k manifest
python forge.py compile --build "$OUT/build/coreai"
FINAL_PHASE=$(python -c 'import json,os; from pathlib import Path; print(json.loads((Path(os.environ["OUT"])/"jeff_tetris_colab/results.json").read_text())["final_phase"])')
"$COREAI_PYTHON" scripts/jeff_lora_parity.py --model "$OUT/merged" --build "$OUT/build/coreai" --rows "$OUT/jeff_tetris_colab/reports/${FINAL_PHASE}_parity_rows.json"
# Require all four argmax matches before continuing. A checkpoint-only recovery may lack parity rows;
# generate reference rows for those exact recovered weights before validating that export.
M=${JEFF_MODELS_DIR:-$HOME/Models/jeff-ane}          # the repo's own convention; base build from scripts/download_model.py base
python forge.py system1-ane-serve --model "$M/base/model" --build "$M/base/coreai" --adapter tetris="$OUT/build/coreai" --port 8796
# In a SECOND terminal (the server stays running):
# Check the saved parity rows against the exported build before accepting ANE results.
# Tune on the 8 dev seeds first; use --split test (32 seeds, 150-piece cap) once, and run the teacher on the same split
python -m projects.tetris_training.play --dataset "$OUT/jeff_tetris_colab/dataset_main" --split dev  --adapter tetris --url http://127.0.0.1:8796 --output "$OUT/evals/dev"
python -m projects.tetris_training.play --dataset "$OUT/jeff_tetris_colab/dataset_main" --split test --adapter tetris --url http://127.0.0.1:8796 --output "$OUT/evals/test"
python -m projects.tetris_training.play --dataset "$OUT/jeff_tetris_colab/dataset_main" --split test --adapter teacher --output "$OUT/evals/teacher-test"
```

Caveat: `--seed 42` makes the 32 test seeds **identical to the repo's already-used benchmark seeds** (a repeated benchmark, not a pristine holdout). For fresh seeds, set `DATA_SEED` to another value above (then the held-out rows also differ from the repo's pilot).

## Appendix (optional, default OFF): Unsloth `FastDecisionModel` variant, UNTESTED for Tetris

Mirrors the Snake A100 recipe (`scripts/unsloth/run_decision_a100.py`; `unsloth==2026.10.3` worked on 2026-10-09 for Snake): `FastDecisionModel` + `DecisionTrainer`, LoRA r16 / alpha 16 (the Snake recipe; the shipped Tetris adapter uses alpha 32), cosine LR 2e-4, seed 3407, base loaded in the appendix's separate mixed-precision settings, Jeff v1.3 **re-keyed** to the `Qwen3_5ForConditionalGeneration` layout exactly as `jeff_folder()` does for Snake.

Read before enabling:
- It trains Unsloth's **Clef joint-schema head with its own prompt template**, not the repo's 255-way readout and serving prompt. Its adapter is **not** a drop-in for the shipped Tetris ANE build and its accuracy is **not comparable** with section 7. Serving it needs the Clef-head path used for the stock Snake build.
- Tetris prompts are ~1,500 tokens (Snake's were far shorter), so `max_seq_length` is 2048 and memory/speed are **unmeasured** here (outside the main path's planning estimate and optimization cutoff).
- `pip install unsloth` replaces `transformers`/`torch` pins in the runtime. Run it **last** (after exporting the main bundle; auto-disconnect must be off) and restart the runtime before re-running the main path.
- Unsloth Tetris numbers, if you run it, are exploratory only.

In [ ]:
RUN_UNSLOTH_VARIANT = False     # opt in. UNTESTED for Tetris.
UNSLOTH_BASE   = "jeff"         # "jeff" = re-keyed Jeff v1.3 (as in the Snake A100 run); "stock" = unsloth/Qwen3.5-0.8B
UNSLOTH_EPOCHS = 2

if RUN_UNSLOTH_VARIANT:
    sh([PY, "-m", "pip", "install", "-q", "unsloth==2026.10.3", "unsloth_zoo"], log=WORK / "pip_unsloth.log")
    (HELP / "run_unsloth_tetris.py").write_text(r"""
# Adapted from Unsloth's "Train your own Decision Model" code, via the Snake A100 recipe (run_decision_a100.py). UNTESTED for Tetris.
import os, json, time, shutil, tarfile
from pathlib import Path
W = Path(os.environ["WORK"]); OUT = W / "unsloth_out"; OUT.mkdir(exist_ok=True)
TAG = os.environ["UNSLOTH_BASE"]; EPOCHS = float(os.environ["UNSLOTH_EPOCHS"]); LOG = {"tag": TAG, "t0": time.time()}
def jeff_folder():
    # same re-key as the Snake recipe: Jeff ships as Qwen3_5Model ('language_model.*', 'visual.*'); Unsloth's head wants
    # Qwen3_5ForConditionalGeneration ('model.language_model.*', 'model.visual.*'); tokenizer/processor from unsloth/Qwen3.5-0.8B.
    from huggingface_hub import snapshot_download
    from safetensors.torch import load_file, save_file
    j = Path(os.environ["BASE_DIR"]); dst = W / "jeff_v13_cg"
    if (dst / "model.safetensors").exists(): return str(dst)
    q = Path(snapshot_download("unsloth/Qwen3.5-0.8B", allow_patterns=["*.json", "*.jinja", "*.txt"]))
    dst.mkdir(parents=True, exist_ok=True)
    for f in q.iterdir():
        if f.name not in ("model.safetensors.index.json", "config.json", "README.md"): shutil.copy(f, dst / f.name)
    cfg = json.loads((j / "config.json").read_text()); cfg["architectures"] = ["Qwen3_5ForConditionalGeneration"]
    (dst / "config.json").write_text(json.dumps(cfg, indent=2))
    sd = load_file(str(j / "model.safetensors"))
    sd = {("model." + k if k.startswith(("language_model.", "visual.")) else k): v for k, v in sd.items()}
    save_file(sd, str(dst / "model.safetensors"), metadata={"format": "pt"})
    return str(dst)
MODEL = jeff_folder() if TAG == "jeff" else "unsloth/Qwen3.5-0.8B"
def convert(path):
    rows = []
    for line in open(path):
        r = json.loads(line)
        rows.append({"state": json.dumps(r["state"]),
                     "questions": json.dumps({"move": {"type": "choice", "instructions": r["instructions"], "criteria": r["options"]}}),
                     "gold": json.dumps({"move": {"type": "choice", "label": r["label"]}})})
    return rows
train_rows, held_rows = convert(os.environ["TRAIN_JSONL"]), convert(os.environ["HELD_JSONL"])
from unsloth import FastDecisionModel, DecisionTrainer, is_bfloat16_supported
from transformers import TrainingArguments
import torch
model, tokenizer = FastDecisionModel.from_pretrained(model_name=MODEL, max_seq_length=2048, load_in_4bit=False, dtype=torch.float16)
model = FastDecisionModel.get_peft_model(model, r=16, lora_alpha=16, lora_dropout=0, use_gradient_checkpointing="unsloth", random_state=3407)
train_items, rep = FastDecisionModel.build_dataset(train_rows, tokenizer, model)
eval_items, rep2 = FastDecisionModel.build_dataset(held_rows, tokenizer, model)
print("build_dataset", rep, rep2, "max len", max(len(i["input_ids"]) for i in train_items), flush=True)
LOG["before"] = FastDecisionModel.evaluate(model, tokenizer, eval_items); print("held-out BEFORE", LOG["before"], flush=True)
trainer = DecisionTrainer(model=model, processing_class=tokenizer, train_dataset=train_items, eval_dataset=eval_items,
    args=TrainingArguments(per_device_train_batch_size=4, gradient_accumulation_steps=8, num_train_epochs=EPOCHS, learning_rate=2e-4,
        lr_scheduler_type="cosine", warmup_steps=10, weight_decay=0.01, bf16=is_bfloat16_supported(), fp16=not is_bfloat16_supported(),
        eval_strategy="no", logging_steps=5, output_dir=str(W / "unsloth_outputs"), save_strategy="no", report_to="none", seed=3407))
t = time.time(); trainer.train(); LOG["train_seconds"] = time.time() - t
LOG["after"] = FastDecisionModel.evaluate(model, tokenizer, eval_items); print("held-out AFTER", LOG["after"], flush=True)
LOG["calibrated"] = FastDecisionModel.calibrate(model, tokenizer, eval_items); print("held-out CALIBRATED", LOG["calibrated"], flush=True)
model.save_pretrained(str(OUT / "adapter_head"))
(OUT / "metrics.json").write_text(json.dumps(LOG, indent=2, default=str))
with tarfile.open(W / "jeff_tetris_unsloth_variant.tgz", "w:gz") as tf: tf.add(OUT, arcname="unsloth_out")
print("DONE", flush=True)
""")
    env = {**os.environ, "WORK": str(WORK), "BASE_DIR": str(BASE), "UNSLOTH_BASE": UNSLOTH_BASE, "UNSLOTH_EPOCHS": str(UNSLOTH_EPOCHS),
           "TRAIN_JSONL": str(RUN / ("phaseB_train.jsonl" if (RUN / "phaseB_train.jsonl").exists() else "phaseA_train.jsonl")),
           "HELD_JSONL": str(MAIN / "validation.jsonl"), "UNSLOTH_CLEF_FAST": "1", "HF_HUB_ENABLE_HF_TRANSFER": "0"}
    sh([PY, HELP / "run_unsloth_tetris.py"], env=env, log=WORK / "unsloth_variant.log")
else:
    print("Unsloth variant not run (RUN_UNSLOTH_VARIANT = False).")
